# Microsoft Agent Framework — Azure OpenAI (Responses API)

In this code sample, you will use the **Microsoft Agent Framework (MAF)** to create a simple agent backed by **Azure OpenAI** using the **Responses API**.

> **Migration note:** This sample previously used Semantic Kernel with GitHub Models. It has been migrated to the Microsoft Agent Framework, and GitHub Models (deprecated, retiring July 2026) has been replaced with Azure OpenAI, which supports the Responses API. The `OpenAIChatClient` in MAF targets Azure OpenAI's stable `/openai/v1/` endpoint and uses the Responses API by default.

The purpose of this sample is to demonstrate the steps that will later be applied in additional code samples when implementing various agentic patterns.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Zaimportuj potrzebne pakiety Pythona


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definiowanie narzędzia

W Microsoft Agent Framework **narzędzie** to zwykła funkcja Pythona dekorowana przez `@tool`, którą agent może wywołać. Poniżej definiujemy narzędzie, które zwraca losowy cel wakacyjny i unika powtarzania poprzedniego.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Tworzenie Agenta

Tutaj tworzymy Agenta o nazwie `TravelAgent`.

W tym przykładzie używamy bardzo podstawowych instrukcji. Śmiało modyfikuj te instrukcje, aby zobaczyć, jak zmienia się zachowanie agenta.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Uruchamianie agenta

Teraz możemy uruchomić agenta. Tworzymy `AgentSession`, aby agent pamiętał rozmowę w kolejnych turach, następnie wysyłamy dwa `user_inputs`. Pierwsze prosi o wycieczkę; drugie mówi, że użytkownikowi nie podobała się propozycja i prosi o inną — agent wykorzystuje historię sesji oraz narzędzie `get_random_destination`, aby odpowiedzieć.

Możesz zmodyfikować te wiadomości, aby obserwować, jak agent reaguje inaczej. Odpowiedzi są **strumieniowane** token po tokenie.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
